# Neev ML Module — Working Demo

**Purpose:** predict 30-day failure risk and show the prediction handoff used by Arnav's optimizer.

Models included: CatBoost failure-risk classifier + HistGradientBoosting 30-day degradation model.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
import joblib

BASE = Path.cwd()
DATA = BASE / 'neev_ml_dataset.csv'
MODEL_DIR = BASE / 'models'
OUTPUT_DIR = BASE / 'output'

df = pd.read_csv(DATA)
print('Dataset:', df.shape)
display(df.head())

## 1. Load Neev's trained models

In [ ]:
risk_model = CatBoostClassifier()
risk_model.load_model(MODEL_DIR / 'neev_failure_risk_model.cbm')

deg_bundle = joblib.load(MODEL_DIR / 'neev_degradation_30d_model.joblib')
deg_model = deg_bundle['model']

print('Risk model:', type(risk_model).__name__)
print('Degradation model:', type(deg_model).__name__)

## 2. Run a real prediction on one asset

In [ ]:
excluded = {'observation_date','asset_id','risk_score','failure_type','failure_severity',
            'maintenance_required','degradation_7d','degradation_14d','degradation_30d',
            'failure_within_30d'}
features = [c for c in df.columns if c not in excluded]

sample = df.iloc[[0]].copy()
p = float(risk_model.predict_proba(sample[features])[:,1][0])
risk_score = p * 100
risk_level = ('LOW' if risk_score <= 30 else 'MODERATE' if risk_score <= 60 else
              'HIGH' if risk_score <= 80 else 'CRITICAL')

print('Asset ID  :', sample['asset_id'].iloc[0])
print('Risk Score:', round(risk_score, 2))
print('Risk Level:', risk_level)

## 3. View Neev → Arnav handoff

In [ ]:
handoff = pd.read_csv(OUTPUT_DIR / 'neev_predictions_for_optimizer.csv')
print('Rows in optimizer handoff:', len(handoff))
display(handoff.head(10))

### Pipeline

`neev_ml_dataset.csv` → **Neev ML models** → `neev_predictions_for_optimizer.csv` → **Arnav OR-Tools optimizer**

The report, feature-importance file, and model files document/support Neev's work; the optimizer consumes the prediction CSV.